# Assignment 1 on Colab

Runtime → Change runtime type → **T4 GPU** (free) or L4/A100 (Pro). Run the cells top to bottom. Everything that must survive a disconnect should be saved to Google Drive: data, checkpoints, results.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/a1'
os.makedirs(WORK, exist_ok=True)
print(WORK)

Mounted at /content/drive
/content/drive/MyDrive/a1


## 1. Code
Clone your repository (or upload the starter kit to Drive once and copy it here).

In [2]:
%cd /content/drive/MyDrive/a1/BLfS-main
%env PYTHONPATH=/content/drive/MyDrive/a1/BLfS-main
!pip -q install regex einops jaxtyping pytest
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

/content/drive/MyDrive/a1/BLfS-main
env: PYTHONPATH=/content/drive/MyDrive/a1/BLfS-main
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.7 MB/s eta 0:00:00
2.11.0+cu130 Tesla T4


## 2. Tests
They run on CPU in a few seconds. Fix failures before training anything.

In [3]:
!python -m pytest -q tests/

........................................................................ [ 97%]
..                                                                       [100%]


## 3. Data
Download the TinyStories files once to Drive (links on Moodle), tokenise them with your tokenizer, then copy `train.bin`/`val.bin` to the local disk for speed at the start of every session.

In [4]:
!mkdir -p /content/data
!cp /content/drive/MyDrive/a1/BLfS-main/data/train.bin /content/data/
!cp /content/drive/MyDrive/a1/BLfS-main/data/valid.bin /content/data/
!ls -la /content/data

total 1067012
drwxr-xr-x 2 root root       4096 Oct 21 19:52 .
drwxr-xr-x 1 root root       4096 Oct 21 19:52 ..
-rw------- 1 root root 1081681224 Oct 21 19:52 train.bin
-rw------- 1 root root   10923504 Oct 21 19:52 valid.bin


## 4. Speed of this GPU (Lecture 2's script)

In [5]:
import torch, time
n, dt = 4096, torch.float16     # bf16 on L4/A100
a = torch.randn(n, n, device='cuda', dtype=dt); b = torch.randn(n, n, device='cuda', dtype=dt)
for _ in range(3): a @ b
torch.cuda.synchronize(); t = time.perf_counter()
for _ in range(20): c = a @ b
torch.cuda.synchronize(); sec = (time.perf_counter() - t) / 20
print(f'{2 * n**3 / sec / 1e12:.1f} TFLOP/s')

23.2 TFLOP/s


In [ ]:
# !PYTHONPATH=. python scripts/train.py --data data/train.bin --val data/valid.bin \
#     --out runs/smoke --steps 200 --eval_every 100 --eval_tokens 65536

## 5. Train
Checkpoints go to Drive (`--out $WORK/runs/...`), so `--resume` continues after a disconnect. Do the resume test on a 200-step run before the reference run.

In [ ]:
# smoke test
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/smoke --steps 200 --eval_every 100 --ckpt_every 100

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
{"step": 0, "loss": 9.26966667175293, "lr": 0.0, "gnorm": 1.1460283994674683, "tokens_per_s": 10697.479867507154}
{"step": 0, "val_loss": 9.265356726944447}
{"step": 50, "loss": 3.955502510070801, "lr": 0.0009051132292283772, "gnorm": 0.3796372711658478, "tokens_per_s": 12880.303808956434}
{"step": 100, "loss": 3.36843204498291, "lr": 0.0005871607054625496, "gnorm": 0.39187225699424744, "tokens_per_s": 18169.992556087393}
{"step": 100, "val_loss": 3.3993036430329084}
{"step": 150, "loss": 3.2442378997802734, "lr": 0.0002452232927684166, "gnorm": 0.39963677525520325, "tokens_per_s": 15316.937301779066}
{"step": 199, "val_loss": 3.090539775788784}
{"lr": 0.001, "steps": 200, "tokens": 1638400, "flops": 183046137

In [ ]:
# # kill the runtime here (Runtime → Interrupt) and rerun with --resume: the log must continue from the last checkpoint
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/smoke --steps 200 --eval_every 100 --ckpt_every 100 --resume

In [ ]:
# the reference run (Table 1 of the handout); 30-60 min on a T4
# !python scripts/train.py --data /content/data/train.bin --val /content/data/val.bin --out $WORK/runs/ref --lr_max 1e-3 --steps 24414 --resume
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/ref --lr_max 1e-3 --steps 24414 --resume

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
{"step": 0, "loss": 9.26966667175293, "lr": 0.0, "gnorm": 1.1460283994674683, "tokens_per_s": 12825.631438713845}
{"step": 0, "val_loss": 9.265356726944447}
{"step": 50, "loss": 8.16170883178711, "lr": 4.098360655737705e-05, "gnorm": 2.303239345550537, "tokens_per_s": 11806.430872236737}
{"step": 100, "loss": 6.243762969970703, "lr": 8.19672131147541e-05, "gnorm": 1.83420991897583, "tokens_per_s": 17126.505029161413}
Traceback (most recent call last):
  File "/content/drive/MyDrive/a1/BLfS-main/scripts/train.py", line 172, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/a1/BLfS-main/scripts/train.py", line 140, in main
    gnorm = clip_grad_norm_(model.parameters(), cfg.grad_clip)  # noqa: F82

In [ ]:
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/resume_test --steps 1000 --ckpt_every 100 --eval_every 100000

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
{"step": 0, "loss": 9.26966667175293, "lr": 0.0, "gnorm": 1.1460283994674683, "tokens_per_s": 12934.276873178751}
{"step": 0, "val_loss": 9.265356726944447}
{"step": 50, "loss": 4.175905704498291, "lr": 0.001, "gnorm": 0.4920872151851654, "tokens_per_s": 12020.312454828994}
{"step": 100, "loss": 3.3632919788360596, "lr": 0.000993862586531225, "gnorm": 0.4142993092536926, "tokens_per_s": 17332.66780539173}
{"step": 150, "loss": 3.150150775909424, "lr": 0.0009756177587652855, "gnorm": 0.4496881067752838, "tokens_per_s": 20175.731249417404}
{"step": 200, "loss": 3.124910831451416, "lr": 0.0009457631880429201, "gnorm": 0.46703004837036133, "tokens_per_s": 22219.670297509976}
{"step": 250, "loss": 2.863547086715698

In [ ]:
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/resume_test --steps 1000 --ckpt_every 100 --eval_every 100000 --resume

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
resumed at step 401
{"step": 450, "loss": 2.3875086307525635, "lr": 0.0006604684692133597, "gnorm": 0.43988513946533203, "tokens_per_s": 31115.14501370843}
{"step": 500, "loss": 2.3671250343322754, "lr": 0.0005871607054625496, "gnorm": 0.41658517718315125, "tokens_per_s": 31244.403283640837}
{"step": 550, "loss": 2.2916183471679688, "lr": 0.0005128392945374505, "gnorm": 0.42139744758605957, "tokens_per_s": 30586.729844431884}
{"step": 600, "loss": 2.280768394470215, "lr": 0.0004395315307866404, "gnorm": 0.4424895644187927, "tokens_per_s": 30781.902577860183}
{"step": 650, "loss": 2.323151111602783, "lr": 0.0003692370589061638, "gnorm": 0.4599281847476959, "tokens_per_s": 30609.528592173927}
{"step": 700, "loss

In [ ]:
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/sweep_3e-4 --lr_max 3e-4 --steps 3000 --eval_every 500 --ckpt_every 1000 --resume

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
{"step": 0, "loss": 9.26966667175293, "lr": 0.0, "gnorm": 1.1460283994674683, "tokens_per_s": 12564.951114069123}
{"step": 0, "val_loss": 9.265356726944447}
{"step": 50, "loss": 7.206651210784912, "lr": 9.999999999999999e-05, "gnorm": 2.344757556915283, "tokens_per_s": 12059.655245831003}
{"step": 100, "loss": 4.7552490234375, "lr": 0.00019999999999999998, "gnorm": 0.5887565612792969, "tokens_per_s": 17361.549288425253}
{"step": 150, "loss": 3.930802345275879, "lr": 0.0002999999999999999, "gnorm": 0.4278930723667145, "tokens_per_s": 20439.284931145787}
{"step": 200, "loss": 3.705223798751831, "lr": 0.00029979500471849756, "gnorm": 0.4588421583175659, "tokens_per_s": 22453.72483697366}
{"step": 250, "loss": 3.3

In [ ]:
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/sweep_1e-3 --lr_max 1e-3 --steps 3000 --eval_every 500 --ckpt_every 1000 --resume

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
{"step": 0, "loss": 9.26966667175293, "lr": 0.0, "gnorm": 1.1460283994674683, "tokens_per_s": 12274.237111774284}
{"step": 0, "val_loss": 9.265356726944447}
{"step": 50, "loss": 5.236014366149902, "lr": 0.0003333333333333334, "gnorm": 0.8057071566581726, "tokens_per_s": 12417.979116382443}
{"step": 100, "loss": 3.6965749263763428, "lr": 0.0006666666666666668, "gnorm": 0.5030551552772522, "tokens_per_s": 17769.68123527988}
{"step": 150, "loss": 3.3452272415161133, "lr": 0.001, "gnorm": 0.42739972472190857, "tokens_per_s": 20795.8856734157}
{"step": 200, "loss": 3.2246031761169434, "lr": 0.0009993166823949922, "gnorm": 0.45515769720077515, "tokens_per_s": 22746.719963317555}
{"step": 250, "loss": 2.9335465431213

In [ ]:
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/sweep_3e-3 --lr_max 3e-3 --steps 3000 --eval_every 500 --ckpt_every 1000 --resume

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
{"step": 0, "loss": 9.26966667175293, "lr": 0.0, "gnorm": 1.1460283994674683, "tokens_per_s": 11928.050109300792}
{"step": 0, "val_loss": 9.265356726944447}
{"step": 50, "loss": 4.175905704498291, "lr": 0.001, "gnorm": 0.4920872151851654, "tokens_per_s": 12404.10895692128}
{"step": 100, "loss": 3.3237760066986084, "lr": 0.002, "gnorm": 0.4257167875766754, "tokens_per_s": 17754.109144881604}
{"step": 150, "loss": 3.1528146266937256, "lr": 0.003, "gnorm": 0.46502184867858887, "tokens_per_s": 20790.431731439094}
{"step": 200, "loss": 3.1238350868225098, "lr": 0.002997950047184977, "gnorm": 0.417581170797348, "tokens_per_s": 22748.440532118137}
{"step": 250, "loss": 2.886319637298584, "lr": 0.002991806414379232, "

In [ ]:
import json, glob
for p in sorted(glob.glob(f"{WORK}/runs/*/summary.json")):
    s = json.load(open(p))
    print(f"{p.split('/')[-2]:15s} lr={s['lr']:<8} steps={s['steps']:<6} "
          f"val_loss={s['val_loss']:.4f} tok/s={s['tokens_per_s']:.0f} mfu={s['mfu']}")

resume_test     lr=0.001    steps=1000   val_loss=2.1825 tok/s=25749 mfu=0.04425690226416266
smoke           lr=0.001    steps=200    val_loss=3.0905 tok/s=14377 mfu=0.024711155392362093
sweep_1e-3      lr=0.001    steps=3000   val_loss=1.7912 tok/s=26728 mfu=0.045940188119807886
sweep_3e-3      lr=0.003    steps=3000   val_loss=1.7794 tok/s=26646 mfu=0.04579857214120595
sweep_3e-4      lr=0.0003   steps=3000   val_loss=2.0721 tok/s=26692 mfu=0.045877974858882016


In [7]:
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/sweep_1e-2 --lr_max 1e-2 --steps 3000 --eval_every 500 --ckpt_every 1000 --resume

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
resumed at step 1001
{"step": 1050, "loss": 2.7514524459838867, "lr": 0.007961266711550924, "gnorm": 0.553830623626709, "tokens_per_s": 31068.463975923096}
{"step": 1100, "loss": 2.7618179321289062, "lr": 0.007750000000000001, "gnorm": 0.41632747650146484, "tokens_per_s": 31168.159084656403}
{"step": 1150, "loss": 2.7316465377807617, "lr": 0.007531900112399003, "gnorm": 0.30411776900291443, "tokens_per_s": 31169.44659768325}
{"step": 1200, "loss": 2.7228994369506836, "lr": 0.007307629410938364, "gnorm": 0.1958225816488266, "tokens_per_s": 31321.08194597249}
{"step": 1250, "loss": 2.794200897216797, "lr": 0.007077868998367396, "gnorm": 0.25618278980255127, "tokens_per_s": 31464.02648337385}
{"step": 1300, "loss

In [9]:
!python scripts/train.py --data /content/data/train.bin --val /content/data/valid.bin --out $WORK/runs/ablate_nowarmup --lr_max 3e-3 --warmup_frac 0 --steps 3000 --eval_every 500 --ckpt_every 1000 --resume

/content/drive/MyDrive/a1/BLfS-main/scripts/train.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))
{"step": 0, "loss": 9.26966667175293, "lr": 0.003, "gnorm": 1.1460283994674683, "tokens_per_s": 12775.326317261062}
{"step": 0, "val_loss": 8.243491746485233}
{"step": 50, "loss": 4.118960380554199, "lr": 0.0029981498719186747, "gnorm": 0.48105520009994507, "tokens_per_s": 11764.127335208235}
{"step": 100, "loss": 3.481868267059326, "lr": 0.002992604558747169, "gnorm": 0.38347792625427246, "tokens_per_s": 17063.3960475523}
{"step": 150, "loss": 3.316737413406372, "lr": 0.002983379259803436, "gnorm": 0.3822774589061737, "tokens_per_s": 20185.060962327607}
{"step": 200, "loss": 3.293529510498047, "lr": 0.0029704992609906377, "gnorm": 0.37889352440834045, "tokens_per_s": 22226.975061459227}
{"step": 250, "loss": 

## 6. results.json
Assemble it from the run summaries and check it before submitting.

In [8]:
!python grading/validate_results.py $WORK/results.json

python3: can't open file '/content/drive/MyDrive/a1/BLfS-main/grading/validate_results.py': [Errno 2] No such file or directory
